---
## Step 0 — Imports

In [ ]:
import ast
import joblib
import numpy as np
import pandas as pd
import re
import os
import spacy
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from scipy.sparse import hstack
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
REPO_ID      = 'bushraasaleem/AI-detection-datsets'
REPO_TYPE    = 'dataset'
MIN_WORDS    = 20



print('Imports OK')

Imports OK


---
## Step 1 — Data loading helpers

In [2]:
def load_pkl(filename, text_col=None, label_col=None, label_val=None, source_name=None):
    """
    Download a pkl from HuggingFace and normalise to {answers, label, source}.
    Auto-detects column names: ans/answer/answers/text and label/target.
    """
    path = hf_hub_download(repo_id=REPO_ID, filename=filename, repo_type=REPO_TYPE)
    df   = pd.read_pickle(path)

    if text_col is None:
        for c in ['answers', 'ans', 'answer', 'text']:
            if c in df.columns:
                text_col = c
                break
    assert text_col in df.columns, f'{filename}: text col not found. Cols: {df.columns.tolist()}'

    if label_col is None:
        for c in ['label', 'target']:
            if c in df.columns:
                label_col = c
                break

    df = df[[text_col] + ([label_col] if label_col and label_col in df.columns else [])].copy()
    df = df.rename(columns={text_col: 'answers'})

    if label_val is not None:
        df['label'] = label_val
    elif label_col and label_col in df.columns:
        df = df.rename(columns={label_col: 'label'})
    else:
        raise ValueError(f'{filename}: cannot determine label. Set label_val explicitly.')

    df['label']  = df['label'].astype(int)
    df['source'] = source_name or filename.replace('.pkl', '')
    df = df[['answers', 'label', 'source']].dropna()
    df['answers'] = df['answers'].astype(str).str.strip().str.lower()
    df = df[df['answers'].str.split().str.len() >= MIN_WORDS]
    df = df.drop_duplicates(subset='answers')
    return df.reset_index(drop=True)


def load_hf_streaming(dataset_name, split, text_extractor, label, source_name, n=6000, **kwargs):
    """
    Load n rows from a HuggingFace dataset in streaming mode.
    text_extractor: callable(row) -> str or None
    """
    ds = load_dataset(dataset_name, split=split, streaming=True, **kwargs)
    texts = []
    for row in ds:
        text = text_extractor(row)
        if isinstance(text, str) and len(text.strip().split()) >= MIN_WORDS:
            texts.append(text.strip().lower())
        if len(texts) >= n:
            break
    df = pd.DataFrame({'answers': texts, 'label': label, 'source': source_name})
    return df.drop_duplicates(subset='answers').reset_index(drop=True)


print('Helpers defined.')

Helpers defined.


---
## Step 2 — Load all sources

### Human sources
| Source | Style | Why included |
|---|---|---|
| HC3 | Reddit Q&A | Casual human |
| WikiText | Encyclopedia | Formal factual |
| OpenWebText | Web articles | Mixed web |
| M4 human | Academic abstracts | Formal academic |
| Reddit ELI5 | Casual explanations | Informal |
| AG News | News articles | Formal news |
| **ArXiv human** | **Academic abstracts** | **Counters M4 AI academic bias** |

### AI sources
| Source | Model | Style |
|---|---|---|
| HC3 | ChatGPT | Reddit-style Q&A |
| M4 AI | Davinci/Dolly/Cohere/Llama | Academic |
| OpenHermes | Instruct models | Chat |
| OpenOrca | GPT-4/3.5 | Instruction following |
| AI 7k | GPT-4o/Mistral/Qwen | Mixed |

In [3]:
print('Loading PKL files from HuggingFace...')

# HC3
try:
    hc3_raw = pd.read_csv('cleaned_data.csv')
except FileNotFoundError:
    path    = hf_hub_download(repo_id=REPO_ID, filename='cleaned_data.csv', repo_type=REPO_TYPE)
    hc3_raw = pd.read_csv(path)

hc3_raw.columns = [c.strip().lower() for c in hc3_raw.columns]
text_col = next((c for c in ['answers','ans','answer','text'] if c in hc3_raw.columns), None)
assert text_col, f'HC3: text col not found. Cols: {hc3_raw.columns.tolist()}'
hc3_raw = hc3_raw.rename(columns={text_col: 'answers'})
hc3_raw['answers'] = hc3_raw['answers'].astype(str).str.strip().str.lower()

hc3_human = (hc3_raw[hc3_raw['label']==0]
             .dropna(subset=['answers'])
             .drop_duplicates(subset='answers')
             .sample(n=min(7500, (hc3_raw['label']==0).sum()), random_state=RANDOM_STATE)
             .assign(source='hc3')[['answers','label','source']])

hc3_ai = (hc3_raw[hc3_raw['label']==1]
          .dropna(subset=['answers'])
          .drop_duplicates(subset='answers')
          .sample(n=min(7500, (hc3_raw['label']==1).sum()), random_state=RANDOM_STATE)
          .assign(source='hc3')[['answers','label','source']])

print(f'  HC3 human: {len(hc3_human):,}   HC3 AI: {len(hc3_ai):,}')

# PKL files
m4_ai    = load_pkl('M4_ai_15k.pkl',         label_val=1, source_name='m4_ai')
m4_human = load_pkl('m4_human_15k.pkl',       label_val=0, source_name='m4_human')
wiki     = load_pkl('wiki_human.pkl',          label_val=0, source_name='wiki')
owt      = load_pkl('openwebtext_human.pkl',   label_val=0, source_name='owt')
hermes   = load_pkl('openhermes.pkl',  text_col='ans', label_col='target', source_name='hermes')
orca     = load_pkl('openorca.pkl',    text_col='ans', label_col='target', source_name='orca')
ai7k     = load_pkl('ai_generated_7.3k.pkl',     label_val=1, source_name='ai7k')
eli5     = load_pkl('reddit_human.pkl',     label_val=0, source_name='eli5')
agnews  =load_pkl('agnews_human.pkl',     label_val=0, source_name='ag_news')
print(f'  M4 AI:        {len(m4_ai):,}')
print(f'  M4 human:     {len(m4_human):,}')
print(f'  Wiki:         {len(wiki):,}')
print(f'  OpenWebText:  {len(owt):,}')
print(f'  OpenHermes:   {len(hermes):,}')
print(f'  OpenOrca:     {len(orca):,}')
print(f'  AI 7k:        {len(ai7k):,}')
print(f'  eli5:        {len(eli5):,}')
print(f'  agnews:        {len(agnews):,}')

Loading PKL files from HuggingFace...
  HC3 human: 7,500   HC3 AI: 7,500
  M4 AI:        14,914
  M4 human:     6,584
  Wiki:         3,678
  OpenWebText:  10,000
  OpenHermes:   5,000
  OpenOrca:     4,500
  AI 7k:        7,303
  eli5:        3,000
  agnews:        2,873


In [4]:
# Extra human sources to balance AI count and fix formal-text bias
print('Loading extra human sources...')


print(f'  AG News:     {len(agnews):,}')

# ArXiv human abstracts — KEY FIX for academic AI bias
# M4 AI is academic text → we need academic HUMAN text to counter it
arxiv_human = load_hf_streaming(
    'scientific_papers', split='train',
    text_extractor=lambda row: row.get('abstract'),
    label=0, source_name='arxiv_human', n=5000,
    name='arxiv'
)
print(f'  ArXiv human: {len(arxiv_human):,}')

Loading extra human sources...
  AG News:     2,873


  ArXiv human: 5,000


---
## Step 3 — Merge, balance and verify

In [5]:
df = pd.concat([
    hc3_human, hc3_ai,
    m4_ai, m4_human,
    wiki, owt,
    hermes, orca, ai7k,
    eli5, agnews, arxiv_human
], ignore_index=True)

df = df.drop_duplicates(subset='answers').reset_index(drop=True)

print('=== Raw merged ===')
print(f'Total rows: {len(df):,}')
print('\nBy source and label:')
print(df.groupby(['source','label']).size().to_string())

n_human = (df['label']==0).sum()
n_ai    = (df['label']==1).sum()
print(f'\nHuman total: {n_human:,}')
print(f'AI total:    {n_ai:,}')

=== Raw merged ===
Total rows: 74,934

By source and label:
source       label
ag_news      0         2873
ai7k         1         7303
arxiv_human  0         5000
eli5         0           82
hc3          0         7500
             1         7500
hermes       1         5000
m4_ai        1        14914
m4_human     0         6584
orca         1         4500
owt          0        10000
wiki         0         3678

Human total: 35,717
AI total:    39,217


In [6]:
# Balance 50/50 while preserving source proportions
n_min = min(n_human, n_ai)
print(f'Balancing to {n_min:,} per class...')

def balanced_sample(df_subset, n, random_state):
    """
    Sample n rows preserving source proportions within the subset.
    """
    total    = len(df_subset)
    sampled  = []
    sources  = df_subset['source'].unique()
    remaining = n
    for i, src in enumerate(sources):
        src_df = df_subset[df_subset['source'] == src]
        if i == len(sources) - 1:
            k = remaining
        else:
            k = round(n * len(src_df) / total)
        k = min(k, len(src_df))
        sampled.append(src_df.sample(n=k, random_state=random_state))
        remaining -= k
    return pd.concat(sampled).reset_index(drop=True)

df_human_bal = balanced_sample(df[df['label']==0], n_min, RANDOM_STATE)
df_ai_bal    = balanced_sample(df[df['label']==1], n_min, RANDOM_STATE)

df = pd.concat([df_human_bal, df_ai_bal]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'\n=== Balanced dataset ===')
print(f'Total: {len(df):,}')
print(f'Label distribution:\n{df["label"].value_counts().to_string()}')
print('\nSource × label breakdown:')
print(df.groupby(['source','label']).size().unstack(fill_value=0).to_string())

Balancing to 35,717 per class...

=== Balanced dataset ===
Total: 71,434
Label distribution:
label
1    35717
0    35717

Source × label breakdown:
label            0      1
source                   
ag_news       2873      0
ai7k             0   6651
arxiv_human   5000      0
eli5            82      0
hc3           7500   6831
hermes           0   4554
m4_ai            0  13583
m4_human      6584      0
orca             0   4098
owt          10000      0
wiki          3678      0


---
## Step 4 — Text cleaning

In [3]:
def clean_text(x):
    """Unwrap serialised list strings; return plain lowercase string."""
    if isinstance(x, list):
        return ' '.join(map(str, x))
    if isinstance(x, dict):
        return ' '.join(map(str, x.values()))
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return ' '.join(map(str, parsed))
            return str(parsed)
        except (ValueError, SyntaxError):
            return x
    return str(x)




In [ ]:
df['answers'] = df['answers'].str.lower().apply(clean_text)
before = len(df)
df = df[df['answers'].str.split().str.len() >= MIN_WORDS].reset_index(drop=True)
print(f'Text cleaning done. Dropped {before - len(df)} short rows. Remaining: {len(df):,}')

---
## Step 5 — Feature extraction




In [4]:
nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])
nlp.add_pipe('sentencizer')


def load_word_list(path):
    with open(path, 'r', encoding='utf-8') as f:
        return set(line.strip().lower() for line in f if line.strip())


chat_words        = load_word_list('chat_words.txt')
function_words    = load_word_list('function_words.txt')
discourse_markers = load_word_list('discourse_markers.txt')

# AI transition words — overused by instruct-tuned models
TRANSITION_WORDS = {
    'furthermore', 'moreover', 'additionally', 'consequently',
    'therefore', 'however', 'nevertheless', 'firstly', 'secondly',
    'thirdly', 'finally', 'in conclusion', 'to summarize',
    'in summary', 'to conclude', 'in addition', 'as a result',
    'on the other hand', 'that being said', 'it is worth noting'
}

# Words AI rarely starts with (casual/personal starts)
FORMAL_STARTS = {
    'the', 'this', 'these', 'it', 'in', 'as', 'furthermore',
    'moreover', 'additionally', 'there', 'one', 'when', 'while'
}


def split_sentences(text):
    doc = nlp(text)
    return [s.text.strip() for s in doc.sents if s.text.strip()]


def extract_features(text):
    """Extract per-sentence features — ratios only, no raw counts."""
    doc          = nlp(text)
    tokens       = [t.text.lower() for t in doc if not t.is_space]
    alpha_tokens = [t.text.lower() for t in doc if t.is_alpha]
    total_tokens = len(tokens)
    total_alpha  = len(alpha_tokens)

    if total_tokens == 0 or total_alpha == 0:
        return {
            'chat_word_ratio': 0.0, 'punct_ratio': 0.0, 'ttr': 0.0,
            'function_word_ratio': 0.0, 'discourse_ratio': 0.0,
            'sentence_length': 0.0
        }

    return {
        'chat_word_ratio'    : sum(1 for t in tokens if t in chat_words)       / total_tokens,
        'punct_ratio'        : sum(1 for t in doc if t.is_punct)               / total_tokens,
        'ttr'                : len(set(alpha_tokens))                           / total_alpha,
        'function_word_ratio': sum(1 for t in tokens if t in function_words)   / total_tokens,
        'discourse_ratio'    : sum(1 for t in tokens if t in discourse_markers) / total_tokens,
        'sentence_length'    : float(total_tokens),
    }


def aggregate_sentence_features(sentences):
    """Aggregate per-sentence features to document level — v3 feature set."""
    feats = [extract_features(s) for s in sentences]
    if not feats:
        return {k: 0.0 for k in [
            'chat_word_ratio', 'punct_ratio', 'ttr', 'function_word_ratio',
            'discourse_ratio', 'sentence_length_std', 'sentence_length_cv',
            'contraction_ratio', 'transition_ratio', 'formal_start_ratio'
        ]}

    feat_df = pd.DataFrame(feats)
    lengths = feat_df['sentence_length'].tolist()
    mean_len = np.mean(lengths) if lengths else 1.0
    std_len  = np.std(lengths)  if len(lengths) > 1 else 0.0

    # Full text for document-level features
    full_text   = ' '.join(sentences)
    all_tokens  = full_text.split()
    total_toks  = max(len(all_tokens), 1)

    # Contraction ratio — AI rarely uses contractions
    contraction_count = sum(1 for t in all_tokens if "'" in t and
                            any(t.endswith(s) for s in ["'t","'re","'ve","'ll","'d","'m","'s"]))

    # Transition word ratio — AI overuses these
    full_lower = full_text.lower()
    transition_count = sum(full_lower.count(tw) for tw in TRANSITION_WORDS)

    # Formal start ratio — how often sentences start formally
    formal_starts = sum(
        1 for s in sentences
        if s.strip() and s.strip().split()[0].lower() in FORMAL_STARTS
    )

    return {
        # --- kept from v2 ---
        'chat_word_ratio'    : feat_df['chat_word_ratio'].mean(),
        'punct_ratio'        : feat_df['punct_ratio'].mean(),
        'ttr'                : feat_df['ttr'].mean(),
        'function_word_ratio': feat_df['function_word_ratio'].mean(),
        'discourse_ratio'    : feat_df['discourse_ratio'].mean(),
        'sentence_length_std': std_len,

        # --- new in v3 (replace sentence_length_mean + avg_word_length) ---
        'sentence_length_cv' : std_len / mean_len if mean_len > 0 else 0.0,
        'contraction_ratio'  : contraction_count / total_toks,
        'transition_ratio'   : transition_count  / total_toks,
        'formal_start_ratio' : formal_starts / max(len(sentences), 1),
    }


def build_features(df_in):
    rows = []
    for _, row in df_in.iterrows():
        sents = split_sentences(row['answers'])
        feats = aggregate_sentence_features(sents)
        feats['label']  = row['label']
        feats['source'] = row['source']
        rows.append(feats)
    return pd.DataFrame(rows)


print('Feature functions defined.')

Feature functions defined.


In [9]:
df_feat = build_features(df)
df.to_pickle("pre.pkl")
df_feat.to_pickle("custom_feat.pkl")
print(f'Feature extraction done. Saved checkpoints.')
print(f'df_feat shape: {df_feat.shape}')

df_feat.head()

Feature extraction done. Saved checkpoints.
df_feat shape: (71289, 12)


,chat_word_ratio,punct_ratio,ttr,function_word_ratio,discourse_ratio,sentence_length_std,sentence_length_cv,contraction_ratio,transition_ratio,formal_start_ratio,label,source
0,0.007723,0.068021,0.769605,0.481954,0.030046,13.731715,0.341585,0.005405,0.010811,0.600000,1,hc3
1,0.001131,0.116517,0.912250,0.514324,0.065434,10.205289,0.465504,0.001754,0.003509,0.307692,0,hc3
2,0.000768,0.088031,0.924462,0.453733,0.048603,6.948180,0.270960,0.000000,0.003086,0.261905,0,owt
3,0.004217,0.167010,0.909682,0.378224,0.046621,7.722687,0.299451,0.002488,0.002488,0.526316,1,m4_ai
4,0.000000,0.087947,0.855976,0.434330,0.068862,11.115555,0.374682,0.003745,0.007491,0.222222,0,wiki


In [5]:
df_feat=pd.read_pickle('custom_feat.pkl')

In [6]:
# Sanity check — mean feature values by label
feature_cols_custom = [c for c in df_feat.columns if c not in ['label','source']]

print('--- Mean feature values by label ---')
print(
    df_feat.groupby('label')[feature_cols_custom]
    .mean().T
    .rename(columns={0: 'Human (0)', 1: 'AI (1)'})
    .round(4)
)

# Key check: transition_ratio and contraction_ratio should differ
means = df_feat.groupby('label')[feature_cols_custom].mean()
print('\n--- Key bias-fix features (should differ between AI and Human) ---')
for feat in ['transition_ratio','contraction_ratio','formal_start_ratio','sentence_length_cv']:
    human_val = means.loc[0, feat]
    ai_val    = means.loc[1, feat]
    diff      = abs(ai_val - human_val)
    print(f'  {feat:25s}  Human={human_val:.4f}  AI={ai_val:.4f}  diff={diff:.4f}')

--- Mean feature values by label ---
label                Human (0)   AI (1)
chat_word_ratio         0.0023   0.0018
punct_ratio             0.1339   0.1394
ttr                     0.8958   0.8738
function_word_ratio     0.3835   0.3742
discourse_ratio         0.0364   0.0392
sentence_length_std    12.9681  12.5144
sentence_length_cv      0.4692   0.4425
contraction_ratio       0.0073   0.0062
transition_ratio        0.0012   0.0022
formal_start_ratio      0.2678   0.3364

--- Key bias-fix features (should differ between AI and Human) ---
  transition_ratio           Human=0.0012  AI=0.0022  diff=0.0009
  contraction_ratio          Human=0.0073  AI=0.0062  diff=0.0011
  formal_start_ratio         Human=0.2678  AI=0.3364  diff=0.0686
  sentence_length_cv         Human=0.4692  AI=0.4425  diff=0.0267


---
## Step 6 — Stratified train/test split

Stratify by **source × label** — every domain is in both train and test.

In [11]:
df_full = pd.concat([
    df[['answers','label','source']].reset_index(drop=True),
    df_feat.drop(columns=['label','source']).reset_index(drop=True)
], axis=1)

df_full['stratum'] = df_full['source'] + '_' + df_full['label'].astype(str)

print('Stratum counts:')
print(df_full['stratum'].value_counts().to_string())

train_df, test_df = train_test_split(
    df_full,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df_full['stratum']
)

print(f'\nTrain: {len(train_df):,}   Test: {len(test_df):,}')
print('\nTrain source×label counts:')
print(train_df.groupby(['source','label']).size().unstack(fill_value=0).to_string())
print('\nTest source×label counts:')
print(test_df.groupby(['source','label']).size().unstack(fill_value=0).to_string())

Stratum counts:
stratum
m4_ai_1          13583
owt_0            10000
hc3_0             7364
hc3_1             6822
ai7k_1            6651
m4_human_0        6584
arxiv_human_0     5000
hermes_1          4554
orca_1            4098
wiki_0            3678
ag_news_0         2873
eli5_0              82

Train: 57,031   Test: 14,258

Train source×label counts:
label           0      1
source                  
ag_news      2299      0
ai7k            0   5321
arxiv_human  4000      0
eli5           66      0
hc3          5891   5458
hermes          0   3643
m4_ai           0  10866
m4_human     5267      0
orca            0   3278
owt          8000      0
wiki         2942      0

Test source×label counts:
label           0     1
source                 
ag_news       574     0
ai7k            0  1330
arxiv_human  1000     0
eli5           16     0
hc3          1473  1364
hermes          0   911
m4_ai           0  2717
m4_human     1317     0
orca            0   820
owt          2000     0
wi

In [12]:
drop_cols = ['answers', 'label', 'source', 'stratum']

X_ans_train    = train_df['answers']
X_ans_test     = test_df['answers']
X_custom_train = train_df.drop(columns=drop_cols)
X_custom_test  = test_df.drop(columns=drop_cols)
y_train        = train_df['label']
y_test         = test_df['label']

# Full dataset for CV
X_ans_all    = df_full['answers']
X_custom_all = df_full.drop(columns=drop_cols)
y_all        = df_full['label']

print(f'Feature columns ({len(X_custom_train.columns)}): {X_custom_train.columns.tolist()}')
print(f'Train: {len(y_train):,}  Test: {len(y_test):,}')
print(f'Train label balance: {y_train.value_counts().to_dict()}')

Feature columns (10): ['chat_word_ratio', 'punct_ratio', 'ttr', 'function_word_ratio', 'discourse_ratio', 'sentence_length_std', 'sentence_length_cv', 'contraction_ratio', 'transition_ratio', 'formal_start_ratio']
Train: 57,031  Test: 14,258
Train label balance: {1: 28566, 0: 28465}


---
## Step 7 — TF-IDF vectorisation

- `max_features=5000` — forces generalisation
- `max_df=0.85` — drops near-universal terms (stop-words for classification)
- `min_df=5` — drops corpus-specific rarities
- Fit on **training data only**

In [13]:
tfidf = TfidfVectorizer(
    max_features=5_000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.85,
    sublinear_tf=True,
)

X_train_tfidf = tfidf.fit_transform(X_ans_train)
X_test_tfidf  = tfidf.transform(X_ans_test)

print(f'TF-IDF vocabulary size: {len(tfidf.vocabulary_):,}')
print(f'Train TF-IDF shape:     {X_train_tfidf.shape}')

TF-IDF vocabulary size: 5,000
Train TF-IDF shape:     (57031, 5000)


---
## Step 8 — Scale custom features and fuse

In [14]:
X_custom_train = X_custom_train.replace([np.inf, -np.inf], 0).fillna(0)
X_custom_test  = X_custom_test.replace([np.inf, -np.inf], 0).fillna(0)

scaler = StandardScaler(with_mean=False)
X_train_custom_scaled = scaler.fit_transform(X_custom_train)
X_test_custom_scaled  = scaler.transform(X_custom_test)

X_train = hstack([X_train_tfidf, X_train_custom_scaled])
X_test  = hstack([X_test_tfidf,  X_test_custom_scaled])

print(f'Final train matrix: {X_train.shape}')
print(f'Final test matrix:  {X_test.shape}')

Final train matrix: (57031, 5010)
Final test matrix:  (14258, 5010)


---
## Step 9 — Train all models and pick the best

Three candidates:
1. **LogisticRegressionCV** — your v2 model, strong baseline
2. **LinearSVC** — better at high-dim sparse text features
3. **LR + SVC soft voting ensemble** — averages both, reduces individual errors

Winner is selected automatically by F1-macro on the held-out test set.

In [15]:
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ── Candidate 1: LogisticRegressionCV ────────────────────────────
print('Training LogisticRegressionCV...')
lr_model = LogisticRegressionCV(
    Cs=10,
    cv=SKF,
    penalty='l2',
    solver='saga',
    max_iter=2000,
    class_weight='balanced',
    scoring='f1_macro',
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
lr_model.fit(X_train, y_train)
best_C = lr_model.C_[0]
print(f'  Best C: {best_C:.4f}')

# ── Candidate 2: LinearSVC (calibrated for predict_proba) ────────
print('Training LinearSVC...')
svc_model = CalibratedClassifierCV(
    LinearSVC(
        C=1.0,
        class_weight='balanced',
        max_iter=2000,
        random_state=RANDOM_STATE
    ),
    cv=5
)
svc_model.fit(X_train, y_train)

# ── Candidate 3: Soft voting ensemble ────────────────────────────
print('Training LR + SVC ensemble...')

# Use fixed-C LR inside ensemble (LogisticRegressionCV not directly votable)
lr_fixed = LogisticRegression(
    C=best_C,
    penalty='l2',
    solver='saga',
    max_iter=2000,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
svc_fixed = CalibratedClassifierCV(
    LinearSVC(
        C=1.0,
        class_weight='balanced',
        max_iter=2000,
        random_state=RANDOM_STATE
    ),
    cv=5
)
ensemble = VotingClassifier(
    estimators=[('lr', lr_fixed), ('svc', svc_fixed)],
    voting='soft',
    n_jobs=-1
)
ensemble.fit(X_train, y_train)

print('\nAll models trained.')

Training LogisticRegressionCV...


C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarnin

  Best C: 1291.5497
Training LinearSVC...
Training LR + SVC ensemble...

All models trained.


In [16]:
# ── Compare all models ───────────────────────────────────────────
candidates = {
    'LogisticRegressionCV': lr_model,
    'LinearSVC':            svc_model,
    'LR + SVC Ensemble':    ensemble,
}

results = {}
best_name, best_f1, best_model = None, 0.0, None

print('=== Model Comparison ===')
print(f'{"Model":25s}  {"Accuracy":>10s}  {"F1-macro":>10s}')
print('-' * 50)

for name, clf in candidates.items():
    pred = clf.predict(X_test)
    acc  = accuracy_score(y_test, pred)
    f1   = f1_score(y_test, pred, average='macro')
    results[name] = {'accuracy': acc, 'f1_macro': f1, 'pred': pred}
    print(f'{name:25s}  {acc:>10.4f}  {f1:>10.4f}')
    if f1 > best_f1:
        best_f1   = f1
        best_name = name
        best_model = clf

print(f'\nBest model: {best_name}  (F1={best_f1:.4f})')
y_pred = results[best_name]['pred']

=== Model Comparison ===
Model                        Accuracy    F1-macro
--------------------------------------------------
LogisticRegressionCV           0.9229      0.9228
LinearSVC                      0.9245      0.9245
LR + SVC Ensemble              0.9267      0.9267

Best model: LR + SVC Ensemble  (F1=0.9267)


---
## Step 10 — Full evaluation of best model

In [17]:
print(f'=== Classification Report — {best_name} ===')
print(classification_report(y_test, y_pred, target_names=['Human', 'AI']))
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))

# Per-source accuracy — reveals which domains are hardest
print('\n=== Per-source accuracy ===')
test_df_eval = test_df.copy()
test_df_eval['pred'] = y_pred
per_source = test_df_eval.groupby('source').apply(
    lambda g: pd.Series({
        'n'        : len(g),
        'accuracy' : round((g['pred'] == g['label']).mean(), 4),
        'human_n'  : (g['label']==0).sum(),
        'ai_n'     : (g['label']==1).sum(),
        'false_pos': ((g['pred']==1) & (g['label']==0)).sum(),  # human predicted as AI
        'false_neg': ((g['pred']==0) & (g['label']==1)).sum(),  # AI predicted as human
    })
)
print(per_source.to_string())
print('\nNote: false_pos = human wrongly flagged as AI (your main bias problem)')
print('      false_neg = AI that slipped through as human')

=== Classification Report — LR + SVC Ensemble ===
              precision    recall  f1-score   support

       Human       0.92      0.94      0.93      7116
          AI       0.93      0.92      0.93      7142

    accuracy                           0.93     14258
   macro avg       0.93      0.93      0.93     14258
weighted avg       0.93      0.93      0.93     14258

Accuracy: 0.9267

Confusion Matrix:
[[6655  461]
 [ 584 6558]]

=== Per-source accuracy ===
                  n  accuracy  human_n    ai_n  false_pos  false_neg
source                                                              
ag_news       574.0    0.9826    574.0     0.0       10.0        0.0
ai7k         1330.0    0.9977      0.0  1330.0        0.0        3.0
arxiv_human  1000.0    0.9310   1000.0     0.0       69.0        0.0
eli5           16.0    1.0000     16.0     0.0        0.0        0.0
hc3          2837.0    0.9489   1473.0  1364.0       81.0       64.0
hermes        911.0    0.9199      0.0   911.0  

C:\Users\HP\AppData\Local\Temp\ipykernel_1940\1734197659.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_source = test_df_eval.groupby('source').apply(


---
## Step 11 — 5-fold cross-validation on full dataset

In [18]:
X_all_tfidf         = tfidf.transform(X_ans_all)
X_all_custom        = X_custom_all.replace([np.inf, -np.inf], 0).fillna(0)
X_all_custom_scaled = scaler.transform(X_all_custom)
X_all               = hstack([X_all_tfidf, X_all_custom_scaled])

# Use fixed-C LR for CV scoring (consistent with best_C found above)
cv_model = LogisticRegression(
    C=best_C,
    penalty='l2',
    solver='saga',
    max_iter=2000,
    class_weight='balanced',
    random_state=RANDOM_STATE,
)

cv_scores = cross_val_score(
    cv_model, X_all, y_all,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='f1_macro',
    n_jobs=-1
)

print('=== 5-Fold Stratified Cross-Validation (F1-macro) ===')
for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {s:.4f}')
print(f'\nMean: {cv_scores.mean():.4f}  ±  {cv_scores.std():.4f}')

gap = abs(accuracy_score(y_test, y_pred) - cv_scores.mean())
print(f'\nGap (test acc vs CV mean F1): {gap:.4f}')
if gap > 0.05:
    print('WARNING: Gap > 5% — possible overfitting.')
else:
    print('OK: Gap small — model generalises well across splits.')

=== 5-Fold Stratified Cross-Validation (F1-macro) ===
  Fold 1: 0.9138
  Fold 2: 0.9124
  Fold 3: 0.9156
  Fold 4: 0.9132
  Fold 5: 0.9147

Mean: 0.9139  ±  0.0011

Gap (test acc vs CV mean F1): 0.0128
OK: Gap small — model generalises well across splits.


---
## Step 12 — Save artifacts

In [23]:
feature_cols_saved = X_custom_train.columns.tolist()

joblib.dump(best_model,         'best_model.pkl')         # winner of comparison
joblib.dump(lr_model,           'logistic_model.pkl')     # LR always saved too
joblib.dump(tfidf,              'tfidf_vectorizer.pkl')
joblib.dump(scaler,             'feature_scaler.pkl')
joblib.dump(feature_cols_saved, 'feature_columns.pkl')

print('Saved:')
for f in ['best_model.pkl','logistic_model.pkl','tfidf_vectorizer.pkl',
          'feature_scaler.pkl','feature_columns.pkl']:
    print(f'  {f}')
print(f'\nBest model: {best_name}')
print(f'Feature columns ({len(feature_cols_saved)}): {feature_cols_saved}')

Saved:
  best_model.pkl
  logistic_model.pkl
  tfidf_vectorizer.pkl
  feature_scaler.pkl
  feature_columns.pkl

Best model: LR + SVC Ensemble
Feature columns (10): ['chat_word_ratio', 'punct_ratio', 'ttr', 'function_word_ratio', 'discourse_ratio', 'sentence_length_std', 'sentence_length_cv', 'contraction_ratio', 'transition_ratio', 'formal_start_ratio']


---
## Step 13 — Inference on new text

Uses `best_model`. Mirrors training exactly — full text, extract once, predict once.

In [27]:
def predict_text(text, model, tfidf, scaler, feature_cols):
    """
    Predict whether text is AI-generated (1) or human-written (0).
    Full text input — no chunking.
    """
    text_clean = clean_text(text.lower())
    sentences  = split_sentences(text_clean)
    feats      = aggregate_sentence_features(sentences)

    feat_df = pd.DataFrame([feats])
    feat_df = feat_df.replace([np.inf, -np.inf], 0).fillna(0)
    feat_df = feat_df.reindex(columns=feature_cols, fill_value=0)

    custom_scaled = scaler.transform(feat_df)
    text_tfidf    = tfidf.transform([text_clean])
    X             = hstack([text_tfidf, custom_scaled])

    prob_ai    = model.predict_proba(X)[0, 1]
    prediction = 1 if prob_ai > 0.5 else 0

    return {
        'prediction' : 'AI' if prediction == 1 else 'Human',
        'prob_ai'    : round(float(prob_ai), 4),
        'prob_human' : round(float(1 - prob_ai), 4),
    }


# Demo — test cases covering the bias scenarios
tests = {
    'Casual human (expect Human)':
        """honestly i just googled it lol. the way i think about it is like,
        imagine you're trying to split a pizza — if you can't do it evenly
        then you need to figure out what the common factor is. might be wrong tho haha""",

    'Formal human essay (expect Human — this was failing in v2)':
        """The relationship between language and cognition has been a subject of
        considerable debate in linguistics and cognitive science. Scholars have long
        argued that the structure of language influences thought. Recent empirical
        studies, however, suggest the relationship is bidirectional and more nuanced
        than previously assumed.""",

    'Typical AI response (expect AI)':
        """Certainly! Here is a comprehensive overview of the topic. Furthermore,
        it is worth noting that machine learning models are trained on large datasets.
        Additionally, the training process involves optimizing parameters to minimize
        the loss function. In conclusion, this approach provides a robust framework
        for solving complex problems.""",

    'Academic AI abstract (expect AI — was failing in v1)':
        """This paper presents a novel framework for neural network optimization.
        We propose a gradient-based approach that achieves state-of-the-art results
        on benchmark datasets. Furthermore, our method demonstrates superior
        performance across multiple evaluation metrics. In conclusion, the proposed
        methodology significantly advances the field.""",
}

print('=== Inference Demo ===')
print()
for desc, text in tests.items():
    result = predict_text(text, best_model, tfidf, scaler, feature_cols_saved)
    status = '✓' if (
        ('Human' in desc and result['prediction'] == 'Human') or
        ('AI' in desc and result['prediction'] == 'AI')
    ) else '✗'
    print(f'{status} {desc}')
    print(f'  → {result}')
    print()

=== Inference Demo ===

✓ Casual human (expect Human)
  → {'prediction': 'Human', 'prob_ai': 0.1098, 'prob_human': 0.8902}

✓ Formal human essay (expect Human — this was failing in v2)
  → {'prediction': 'Human', 'prob_ai': 0.4984, 'prob_human': 0.5016}

✓ Typical AI response (expect AI)
  → {'prediction': 'AI', 'prob_ai': 0.9999, 'prob_human': 0.0001}

✓ Academic AI abstract (expect AI — was failing in v1)
  → {'prediction': 'AI', 'prob_ai': 0.9935, 'prob_human': 0.0065}



---
## Step 14 — Real-world validation

Paste your own text or text from current AI tools (GPT-4o, Claude, Gemini).
This is the ultimate check — run this on real examples before trusting the model.

In [51]:
my_text = """Furthermore, it is worth noting that this approach provides 
comprehensive solutions. In conclusion

."""

result = predict_text(my_text, best_model, tfidf, scaler, feature_cols_saved)
print(f'Prediction: {result["prediction"]}')
print(f'P(AI):      {result["prob_ai"]}')
print(f'P(Human):   {result["prob_human"]}')

Prediction: AI
P(AI):      1.0
P(Human):   0.0


---
## Step 15 — Load from checkpoint on future runs

On any future rerun, start from here to skip feature extraction entirely.

In [9]:
# Run this cell on future reruns instead of Steps 2-5
# Loads saved model + artifacts — ready to predict immediately

best_model         = joblib.load('best_model.pkl')
tfidf              = joblib.load('tfidf_vectorizer.pkl')
scaler             = joblib.load('feature_scaler.pkl')
feature_cols_saved = joblib.load('feature_columns.pkl')

print('All artifacts loaded.')
print(f'Feature columns: {feature_cols_saved}')

# Test it immediately
test_text = "Furthermore, it is worth noting that this approach provides comprehensive solutions."
print(f'\nQuick test: {predict_text(test_text, best_model, tfidf, scaler, feature_cols_saved)}')

All artifacts loaded.
Feature columns: ['chat_word_ratio', 'punct_ratio', 'ttr', 'function_word_ratio', 'discourse_ratio', 'sentence_length_std', 'sentence_length_cv', 'contraction_ratio', 'transition_ratio', 'formal_start_ratio']

Quick test: {'prediction': 'AI', 'prob_ai': 1.0, 'prob_human': 0.0}
